# Silver — Food Inspections (dev)

Type-conversion layer only: reads the bronze Parquet (all STRING), **infers each column's type from the bronze data**, and enforces that schema on every row. The only quality gates are **null checks** (before conversion, and after conversion to prove no value was lost). No cleaning, filtering, or dedup. Output lands in `silver/food_inspections/<ingest-date>/`.

In [0]:
# Source and destination paths (bronze -> silver, same ingest-date folder convention)
from datetime import date

INGEST_DATE = date.today().isoformat()
BRONZE_PATH = f"/Volumes/workspace/foodguard_ai/bronze/food_inspections/{INGEST_DATE}"
SILVER_BASE = "/Volumes/workspace/foodguard_ai/silver/food_inspections"
TARGET_PATH = f"{SILVER_BASE}/{INGEST_DATE}"

print(f"Bronze : {BRONZE_PATH}")
print(f"Target : {TARGET_PATH}")

In [0]:
# Read the bronze Parquet (raw, all STRING — as landed by the bronze notebook)
bronze_df = spark.read.parquet(BRONZE_PATH)

bronze_df.printSchema()
print(f"Rows: {bronze_df.count()}")

In [0]:
# Null check (1 of 3): null counts per column in BRONZE, before any conversion.
from pyspark.sql import functions as F

nulls_bronze_row = bronze_df.agg(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in bronze_df.columns]
).first()

nulls_bronze = [(c, nulls_bronze_row[c]) for c in bronze_df.columns]
display(spark.createDataFrame(nulls_bronze, ["column", "bronze_nulls"]))

In [0]:
# Infer the schema FROM THE BRONZE DATA: a candidate type is adopted only if
# EVERY non-null value in the column casts to it without producing a null
# (i.e. zero conversion failures across all rows). All-null columns stay STRING.
# try_cast is ANSI-safe: a failed conversion becomes NULL instead of raising.
CANDIDATES = ["long", "double", "timestamp", "date", "boolean"]

def infer_type(df, col):
    agg = df.agg(
        F.sum(F.col(col).isNotNull().cast("int")).alias("non_null"),
        *[
            F.sum((F.col(col).isNotNull() & F.expr(f"try_cast(`{col}` as {t})").isNull()).cast("int")).alias(t)
            for t in CANDIDATES
        ],
    ).first()
    if agg["non_null"] == 0:
        return "string"
    for t in CANDIDATES:
        if agg[t] == 0:
            return t
    return "string"

inferred = {c: infer_type(bronze_df, c) for c in bronze_df.columns}

print("Inferred schema (enforced on all rows):")
for c, t in inferred.items():
    print(f"  {c:20s} -> {t}")

In [0]:
# Enforce the inferred schema on ALL rows (ANSI-safe try_cast — a failed conversion becomes NULL, flagged by the null check below).
silver_df = bronze_df.select([F.expr(f"try_cast(`{c}` as {inferred[c]})").alias(c) for c in bronze_df.columns])

silver_df.printSchema()

# Null check (2 of 3): rows whose conversion FAILED would surface as new nulls.
# Any value that was non-null in bronze but null in silver is a coercion failure.
nulls_after_row = silver_df.agg(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in silver_df.columns]
).first()

coercion_failures = [
    (c, inferred[c], nulls_after_row[c] - nulls_bronze_row[c])
    for c in silver_df.columns
    if inferred[c] != "string" and nulls_after_row[c] > nulls_bronze_row[c]
]

if coercion_failures:
    display(spark.createDataFrame(coercion_failures, ["column", "enforced_type", "conversion_failures"]))
else:
    print("All enforced casts succeeded on every row — no value lost to conversion.")

In [0]:
# Write as Parquet: single part file per daily folder, overwrite on re-run.
(
    silver_df.coalesce(1)
    .write
    .mode("overwrite")
    .parquet(TARGET_PATH)
)

print(f"Written to {TARGET_PATH}")

In [0]:
# Null check (3 of 3) + final sanity: re-read the silver Parquet and confirm
# row count, enforced schema, and that nulls are unchanged (conversion lost nothing).
check_df = spark.read.parquet(TARGET_PATH)

src_count, out_count = bronze_df.count(), check_df.count()
same_schema = silver_df.schema == check_df.schema

nulls_out_row = check_df.agg(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in check_df.columns]
).first()
nulls_match = all(nulls_out_row[c] == nulls_bronze_row[c] for c in check_df.columns)

report = [(c, nulls_bronze_row[c], nulls_out_row[c]) for c in check_df.columns]
display(spark.createDataFrame(report, ["column", "bronze_nulls", "silver_nulls"]))

print(f"Rows  — bronze: {src_count} | silver: {out_count} | match: {src_count == out_count}")
print(f"Schema match (enforced types persisted): {same_schema}")
print(f"Nulls unchanged across conversion: {nulls_match}")
assert src_count == out_count, "Row count mismatch!"
assert same_schema, "Schema mismatch — enforced types were not persisted!"
assert nulls_match, "Null counts changed — conversion altered data!"
display(check_df.limit(5))